In [0]:
import pandas as pd
from pyspark.sql import SparkSession

In [0]:
spark = SparkSession.builder.getOrCreate()
tables = ['customers', 'sales', 'sales_orders']
base_url = 'https://raw.githubusercontent.com/Bhevendra/ML-Datasets/refs/heads/main/retail_data'
base_path = "/Volumes/ecommerce_analytics/bronze/ingestion_raw/"

for table in tables:
    print("PRoessing the table ", table)
    file_url = f"{base_url}/{table}.csv"
    print("Processing the URL", file_url)

    df = pd.read_csv(file_url)
    output_path = f"{base_path}/{table}"
    spark_df = spark.createDataFrame(df)
    spark_df.write.format("csv").mode("overwrite").save(output_path)
print ("All tables are processed")
    

In [0]:

display(dbutils.fs.ls("/Volumes/ecommerce_analytics/bronze/ingestion_raw/"))

In [0]:
spark.sql("DROP TABLE IF EXISTS ecommerce_analytics.bronze.customers")
spark.sql("DROP TABLE IF EXISTS ecommerce_analytics.bronze.sales")
spark.sql("DROP TABLE IF EXISTS ecommerce_analytics.bronze.sales_orders")

In [0]:
spark.sql("DROP TABLE IF EXISTS ecommerce_analytics.bronze.customers")
spark.sql("DROP TABLE IF EXISTS ecommerce_analytics.bronze.sales")
spark.sql("DROP TABLE IF EXISTS ecommerce_analytics.bronze.sales_orders")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, col
from pyspark.sql.functions import input_file_name
spark = SparkSession.builder.getOrCreate()
tables = ["customers","sales","sales_orders"]
source_path = "/Volumes/ecommerce_analytics/bronze/raw_data/"
catalog = "ecommerce_analytics"
schema = "bronze"

for table in tables:
    print(f"Processing table {table}")
    input_path = f"{source_path}{table}"
    df = spark.read.csv(input_path, inferSchema=True, header=True)
    print(f'Completed reading for the table {table}, Now adding meta columns')
    df = df.withColumn("last_update_ts", current_timestamp()) \
             .withColumn("file_path", col("_metadata.file_path"))
    print (f'Added metadata columns sucessfully')

    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{catalog}.{schema}.{table}")
    print(f"Completed for {table}")
print(f"Write completed for all {table}")